# Model Deployment and Productionization

## Introduction

In previous notebooks, we've explored the Ames housing dataset, performed data preprocessing, feature engineering, and built various predictive models. Now, we'll focus on how to take our best model and deploy it for practical use.

This notebook covers:
1. Creating a reusable prediction pipeline
2. Model serialization (saving and loading)
3. Building a simple API for predictions
4. Creating a basic web interface

These steps are crucial for transforming a machine learning model from a research project into a usable product that can provide value in real-world scenarios.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

In [ ]:
# Load our preprocessed data
try:
    # Try to use IPython magic if in notebook environment
    get_ipython().run_line_magic('run', 'src/preprocessing.py')
except (NameError, AttributeError):
    # If not in notebook environment, import directly
    import sys
    import os
    sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))
    from preprocessing import (
        dataset_2, target_2,
        housing_df  # Original dataframe before preprocessing
    )

## 1. Creating a Reusable Prediction Pipeline

A key step in productionizing a machine learning model is to create a pipeline that encapsulates all preprocessing steps and the model itself. This ensures that new data goes through the exact same transformations as the training data.

In [ ]:
# Load the original data to build a pipeline from scratch
try:
    from src.load_data_02 import load_data
    housing_df = load_data()
except (ImportError, ModuleNotFoundError):
    print("Could not load data from src/load_data_02.py")
    # If we can't load the original data, we'll use a simplified approach with our preprocessed data
    print("Using preprocessed data instead")

In [ ]:
# Check if we have the original data
if 'housing_df' in locals() and not housing_df.empty:
    print(f"Original data shape: {housing_df.shape}")
    
    # Identify numeric and categorical columns
    numeric_cols = housing_df.select_dtypes(include=['number']).columns.tolist()
    numeric_cols.remove('SalePrice')  # Remove target variable
    
    categorical_cols = housing_df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"Number of numeric features: {len(numeric_cols)}")
    print(f"Number of categorical features: {len(categorical_cols)}")
    
    # Create preprocessing pipelines for numeric and categorical features
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    
    # Combine preprocessing steps
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols)
        ])
    
    # Create the full pipeline with preprocessing and model
    full_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', Lasso(alpha=100, max_iter=10000))
    ])
    
    # Split the data
    X = housing_df.drop('SalePrice', axis=1)
    y = housing_df['SalePrice']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Fit the pipeline
    print("Fitting the pipeline...")
    full_pipeline.fit(X_train, y_train)
    
    # Evaluate the pipeline
    train_score = full_pipeline.score(X_train, y_train)
    test_score = full_pipeline.score(X_test, y_test)
    
    print(f"Training R² score: {train_score:.4f}")
    print(f"Test R² score: {test_score:.4f}")
else:
    # If we don't have the original data, use our preprocessed data
    print("Using preprocessed data to create a simplified pipeline")
    
    # Split the preprocessed data
    X_train, X_test, y_train, y_test = train_test_split(dataset_2, target_2, test_size=0.2, random_state=42)
    
    # Create a simplified pipeline with just the model
    full_pipeline = Pipeline(steps=[
        ('model', Lasso(alpha=100, max_iter=10000))
    ])
    
    # Fit the pipeline
    print("Fitting the pipeline...")
    full_pipeline.fit(X_train, y_train)
    
    # Evaluate the pipeline
    train_score = full_pipeline.score(X_train, y_train)
    test_score = full_pipeline.score(X_test, y_test)
    
    print(f"Training R² score: {train_score:.4f}")
    print(f"Test R² score: {test_score:.4f}")

### Feature Selection in the Pipeline

Based on our feature selection analysis in the previous notebook, we can create a more efficient pipeline that uses only the most important features.

In [ ]:
# Create a list of important features based on our previous analysis
important_features = [
    # Overall quality features
    'OverallQual_7', 'OverallQual_8', 'OverallQual_9', 'OverallQual_10',
    # PCA components
    'PC 1', 'PC 2', 'PC 3',
    # Quality features
    'ExterQual_Ex', 'KitchenQual_Ex',
    # Size and area features
    'GrLivArea', 'TotalBsmtSF',
    # Garage features
    'GarageCars_3',
    # Bathroom features
    'FullBath_1', 'FullBath_2',
    # Neighborhood features
    'Neighborhood_NoRidge', 'Neighborhood_StoneBr',
    # Other important features
    'BsmtQual_Ex', 'YearBuilt'
]

# Check which features are actually in our dataset
available_features = [f for f in important_features if f in dataset_2.columns]
print(f"Using {len(available_features)} important features out of {len(important_features)} identified")

# Create a pipeline with feature selection
optimized_pipeline = Pipeline(steps=[
    ('model', Lasso(alpha=100, max_iter=10000))
])

# Fit the pipeline on the selected features
X_train_selected = X_train[available_features]
X_test_selected = X_test[available_features]

print("Fitting the optimized pipeline...")
optimized_pipeline.fit(X_train_selected, y_train)

# Evaluate the optimized pipeline
train_score_opt = optimized_pipeline.score(X_train_selected, y_train)
test_score_opt = optimized_pipeline.score(X_test_selected, y_test)

print(f"Training R² score (optimized): {train_score_opt:.4f}")
print(f"Test R² score (optimized): {test_score_opt:.4f}")
print(f"Difference in test score: {test_score_opt - test_score:.4f}")

## 2. Model Serialization

Once we have a trained pipeline, we need to save it so it can be loaded and used later without retraining. This is known as model serialization.

In [ ]:
# Create a directory for saving models if it doesn't exist
import os
models_dir = 'models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"Created directory: {models_dir}")

In [ ]:
# Save the full pipeline using pickle
with open(os.path.join(models_dir, 'full_pipeline.pkl'), 'wb') as f:
    pickle.dump(full_pipeline, f)
print("Saved full pipeline using pickle")

# Save the optimized pipeline using joblib (better for large NumPy arrays)
joblib.dump(optimized_pipeline, os.path.join(models_dir, 'optimized_pipeline.joblib'))
print("Saved optimized pipeline using joblib")

# Also save the list of important features
with open(os.path.join(models_dir, 'important_features.pkl'), 'wb') as f:
    pickle.dump(available_features, f)
print("Saved list of important features")

In [ ]:
# Load the models back to verify they work
with open(os.path.join(models_dir, 'full_pipeline.pkl'), 'rb') as f:
    loaded_full_pipeline = pickle.load(f)
print("Loaded full pipeline")

loaded_optimized_pipeline = joblib.load(os.path.join(models_dir, 'optimized_pipeline.joblib'))
print("Loaded optimized pipeline")

with open(os.path.join(models_dir, 'important_features.pkl'), 'rb') as f:
    loaded_features = pickle.load(f)
print("Loaded important features")

# Verify the loaded models work correctly
loaded_full_score = loaded_full_pipeline.score(X_test, y_test)
loaded_opt_score = loaded_optimized_pipeline.score(X_test_selected, y_test)

print(f"Loaded full pipeline test R² score: {loaded_full_score:.4f}")
print(f"Loaded optimized pipeline test R² score: {loaded_opt_score:.4f}")

# Check if the scores match
assert abs(loaded_full_score - test_score) < 1e-10, "Full pipeline scores don't match"
assert abs(loaded_opt_score - test_score_opt) < 1e-10, "Optimized pipeline scores don't match"
print("Verification successful: loaded models produce the same scores as original models")

## 3. Building a Simple API for Predictions

Now that we have a serialized model, we can create a simple API to serve predictions. We'll use Flask, a lightweight web framework for Python.

In [ ]:
# Create a Flask API file
api_code = '''
from flask import Flask, request, jsonify
import pandas as pd
import numpy as np
import pickle
import joblib
import os

app = Flask(__name__)

# Load the model and important features
models_dir = 'models'
with open(os.path.join(models_dir, 'important_features.pkl'), 'rb') as f:
    important_features = pickle.load(f)

optimized_pipeline = joblib.load(os.path.join(models_dir, 'optimized_pipeline.joblib'))

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Get JSON data from request
        data = request.get_json()
        
        # Convert to DataFrame
        input_df = pd.DataFrame(data, index=[0])
        
        # Check if we have all required features
        missing_features = [f for f in important_features if f not in input_df.columns]
        if missing_features:
            # For missing features, add them with zeros (one-hot encoding default)
            for feature in missing_features:
                input_df[feature] = 0
        
        # Select only the important features in the right order
        input_df = input_df[important_features]
        
        # Make prediction
        prediction = optimized_pipeline.predict(input_df)[0]
        
        # Return prediction
        return jsonify({
            'prediction': float(prediction),
            'formatted_prediction': f"${prediction:,.2f}"
        })
    
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy'})

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)
'''

# Write the API code to a file
with open('api.py', 'w') as f:
    f.write(api_code)
print("Created API file: api.py")

In [ ]:
# Create a test script to call the API
test_api_code = '''
import requests
import json
import pandas as pd
import sys
import os

# Load a sample from the test set
try:
    # Try to use the preprocessed data
    sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))
    from preprocessing import dataset_2, target_2
    
    # Get a sample house
    sample = dataset_2.iloc[0].to_dict()
    actual_price = float(target_2.iloc[0])
    
    # Make a prediction request
    response = requests.post(
        'http://localhost:5000/predict',
        json=sample
    )
    
    # Print the response
    result = response.json()
    print(f"API Response: {result}")
    print(f"Actual price: ${actual_price:,.2f}")
    print(f"Prediction error: ${abs(result['prediction'] - actual_price):,.2f}")
    
except Exception as e:
    print(f"Error: {e}")
    print("Make sure the API server is running with 'python api.py'")
'''

# Write the test API code to a file
with open('test_api.py', 'w') as f:
    f.write(test_api_code)
print("Created test API file: test_api.py")

### Running the API

To run the API, you would execute the following commands in a terminal:

```bash
# Install Flask if needed
pip install flask

# Start the API server
python api.py
```

Then, in another terminal, you can test the API:

```bash
# Install requests if needed
pip install requests

# Test the API
python test_api.py
```

## 4. Creating a Basic Web Interface

Finally, let's create a simple web interface for our housing price predictor using HTML, CSS, and JavaScript.

In [ ]:
# Create a directory for the web interface
web_dir = 'web'
if not os.path.exists(web_dir):
    os.makedirs(web_dir)
    print(f"Created directory: {web_dir}")

In [ ]:
# Create an HTML file for the web interface
html_code = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Ames Housing Price Predictor</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            line-height: 1.6;
            margin: 0;
            padding: 20px;
            color: #333;
            max-width: 800px;
            margin: 0 auto;
        }
        h1 {
            color: #2c3e50;
            text-align: center;
            margin-bottom: 30px;
        }
        .form-group {
            margin-bottom: 15px;
        }
        label {
            display: block;
            margin-bottom: 5px;
            font-weight: bold;
        }
        input, select {
            width: 100%;
            padding: 8px;
            border: 1px solid #ddd;
            border-radius: 4px;
        }
        button {
            background-color: #3498db;
            color: white;
            border: none;
            padding: 10px 15px;
            border-radius: 4px;
            cursor: pointer;
            font-size: 16px;
        }
        button:hover {
            background-color: #2980b9;
        }
        .result {
            margin-top: 20px;
            padding: 15px;
            background-color: #f8f9fa;
            border-radius: 4px;
            text-align: center;
        }
        .price {
            font-size: 24px;
            font-weight: bold;
            color: #2c3e50;
        }
        .error {
            color: #e74c3c;
            text-align: center;
        }
        .features {
            display: grid;
            grid-template-columns: repeat(2, 1fr);
            gap: 15px;
        }
    </style>
</head>
<body>
    <h1>Ames Housing Price Predictor</h1>
    
    <div class="form-container">
        <div class="features">
            <div class="form-group">
                <label for="overallQual">Overall Quality (1-10)</label>
                <select id="overallQual">
                    <option value="1">1 (Very Poor)</option>
                    <option value="2">2 (Poor)</option>
                    <option value="3">3 (Fair)</option>
                    <option value="4">4 (Below Average)</option>
                    <option value="5">5 (Average)</option>
                    <option value="6">6 (Above Average)</option>
                    <option value="7" selected>7 (Good)</option>
                    <option value="8">8 (Very Good)</option>
                    <option value="9">9 (Excellent)</option>
                    <option value="10">10 (Very Excellent)</option>
                </select>
            </div>
            
            <div class="form-group">
                <label for="grLivArea">Above Ground Living Area (sqft)</label>
                <input type="number" id="grLivArea" value="1500">
            </div>
            
            <div class="form-group">
                <label for="totalBsmtSF">Total Basement Area (sqft)</label>
                <input type="number" id="totalBsmtSF" value="1000">
            </div>
            
            <div class="form-group">
                <label for="yearBuilt">Year Built</label>
                <input type="number" id="yearBuilt" value="1980">
            </div>
            
            <div class="form-group">
                <label for="fullBath">Full Bathrooms</label>
                <select id="fullBath">
                    <option value="1">1</option>
                    <option value="2" selected>2</option>
                    <option value="3">3</option>
                    <option value="4">4+</option>
                </select>
            </div>
            
            <div class="form-group">
                <label for="garageCars">Garage Cars</label>
                <select id="garageCars">
                    <option value="0">0 (No Garage)</option>
                    <option value="1">1</option>
                    <option value="2" selected>2</option>
                    <option value="3">3</option>
                    <option value="4">4+</option>
                </select>
            </div>
            
            <div class="form-group">
                <label for="neighborhood">Neighborhood</label>
                <select id="neighborhood">
                    <option value="NAmes">North Ames</option>
                    <option value="Edwards">Edwards</option>
                    <option value="BrkSide">Brookside</option>
                    <option value="NoRidge" selected>Northridge</option>
                    <option value="StoneBr">Stone Brook</option>
                    <option value="NridgHt">Northridge Heights</option>
                    <option value="Other">Other</option>
                </select>
            </div>
            
            <div class="form-group">
                <label for="exterQual">Exterior Quality</label>
                <select id="exterQual">
                    <option value="Po">Poor</option>
                    <option value="Fa">Fair</option>
                    <option value="TA">Typical/Average</option>
                    <option value="Gd">Good</option>
                    <option value="Ex" selected>Excellent</option>
                </select>
            </div>
        </div>
        
        <div class="form-group" style="margin-top: 20px;">
            <button id="predictBtn">Predict Price</button>
        </div>
    </div>
    
    <div id="result" class="result" style="display: none;">
        <h2>Estimated House Price</h2>
        <div id="price" class="price"></div>
    </div>
    
    <div id="error" class="error" style="display: none;"></div>
    
    <script>
        document.getElementById('predictBtn').addEventListener('click', async function() {
            // Hide previous results and errors
            document.getElementById('result').style.display = 'none';
            document.getElementById('error').style.display = 'none';
            
            try {
                // Get input values
                const overallQual = parseInt(document.getElementById('overallQual').value);
                const grLivArea = parseFloat(document.getElementById('grLivArea').value);
                const totalBsmtSF = parseFloat(document.getElementById('totalBsmtSF').value);
                const yearBuilt = parseInt(document.getElementById('yearBuilt').value);
                const fullBath = parseInt(document.getElementById('fullBath').value);
                const garageCars = parseInt(document.getElementById('garageCars').value);
                const neighborhood = document.getElementById('neighborhood').value;
                const exterQual = document.getElementById('exterQual').value;
                
                // Create one-hot encoded features
                const data = {
                    'GrLivArea': grLivArea,
                    'TotalBsmtSF': totalBsmtSF,
                    'YearBuilt': yearBuilt
                };
                
                // Add one-hot encoded features
                for (let i = 1; i <= 10; i++) {
                    data[`OverallQual_${i}`] = overallQual === i ? 1 : 0;
                }
                
                for (let i = 1; i <= 4; i++) {
                    data[`FullBath_${i}`] = fullBath === i ? 1 : 0;
                }
                
                for (let i = 0; i <= 4; i++) {
                    data[`GarageCars_${i}`] = garageCars === i ? 1 : 0;
                }
                
                // Neighborhood features
                const neighborhoods = ['NAmes', 'Edwards', 'BrkSide', 'NoRidge', 'StoneBr', 'NridgHt'];
                neighborhoods.forEach(n => {
                    data[`Neighborhood_${n}`] = neighborhood === n ? 1 : 0;
                });
                
                // Exterior quality features
                const qualities = ['Po', 'Fa', 'TA', 'Gd', 'Ex'];
                qualities.forEach(q => {
                    data[`ExterQual_${q}`] = exterQual === q ? 1 : 0;
                });
                
                // Make API request
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json'
                    },
                    body: JSON.stringify(data)
                });
                
                if (!response.ok) {
                    throw new Error('API request failed');
                }
                
                const result = await response.json();
                
                // Display the result
                document.getElementById('price').textContent = result.formatted_prediction;
                document.getElementById('result').style.display = 'block';
                
            } catch (error) {
                // Display error message
                document.getElementById('error').textContent = `Error: ${error.message}`;
                document.getElementById('error').style.display = 'block';
                console.error('Error:', error);
            }
        });
    </script>
</body>
</html>
'''

# Write the HTML code to a file
with open(os.path.join(web_dir, 'index.html'), 'w') as f:
    f.write(html_code)
print(f"Created web interface: {os.path.join(web_dir, 'index.html')}")

In [ ]:
# Update the API to serve the web interface
updated_api_code = '''
from flask import Flask, request, jsonify, send_from_directory
import pandas as pd
import numpy as np
import pickle
import joblib
import os

app = Flask(__name__)

# Load the model and important features
models_dir = 'models'
with open(os.path.join(models_dir, 'important_features.pkl'), 'rb') as f:
    important_features = pickle.load(f)

optimized_pipeline = joblib.load(os.path.join(models_dir, 'optimized_pipeline.joblib'))

@app.route('/')
def index():
    return send_from_directory('web', 'index.html')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Get JSON data from request
        data = request.get_json()
        
        # Convert to DataFrame
        input_df = pd.DataFrame(data, index=[0])
        
        # Check if we have all required features
        missing_features = [f for f in important_features if f not in input_df.columns]
        if missing_features:
            # For missing features, add them with zeros (one-hot encoding default)
            for feature in missing_features:
                input_df[feature] = 0
        
        # Select only the important features in the right order
        input_df = input_df[important_features]
        
        # Make prediction
        prediction = optimized_pipeline.predict(input_df)[0]
        
        # Return prediction
        return jsonify({
            'prediction': float(prediction),
            'formatted_prediction': f"${prediction:,.2f}"
        })
    
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy'})

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)
'''

# Write the updated API code to a file
with open('api.py', 'w') as f:
    f.write(updated_api_code)
print("Updated API file to serve the web interface")

### Running the Web Interface

To run the web interface, you would execute the following command in a terminal:

```bash
# Start the API server with web interface
python api.py
```

Then, open a web browser and navigate to `http://localhost:5000` to use the interface.

## 5. Deployment Considerations

For a production deployment, you would need to consider several additional factors:

1. **Security**: Implement authentication, input validation, and protection against common web vulnerabilities
2. **Scalability**: Use a production-ready web server like Gunicorn or uWSGI behind a reverse proxy like Nginx
3. **Monitoring**: Set up logging and monitoring to track usage and detect issues
4. **Containerization**: Package the application in a Docker container for easier deployment
5. **Cloud Deployment**: Deploy to a cloud platform like AWS, Google Cloud, or Azure
6. **CI/CD**: Implement continuous integration and deployment pipelines
7. **Model Versioning**: Set up a system to track model versions and performance
8. **Model Retraining**: Establish a process for periodically retraining the model with new data

These considerations are beyond the scope of this notebook but are crucial for a real-world deployment.

## Summary

In this notebook, we've covered the essential steps for deploying a machine learning model:

1. **Creating a Reusable Prediction Pipeline**: We built a pipeline that encapsulates all preprocessing steps and the model itself, ensuring consistent predictions on new data.

2. **Model Serialization**: We saved our trained model to disk using both pickle and joblib, allowing it to be loaded and used without retraining.

3. **Building a Simple API**: We created a Flask API that serves predictions based on input data, making our model accessible to other applications.

4. **Creating a Basic Web Interface**: We developed a simple web interface that allows users to input housing features and get price predictions.

These steps transform our machine learning model from a research project into a usable product that can provide value in real-world scenarios. In the next notebook, we'll explore how to apply our model to real-world business problems and quantify its business value.